# 08 · Capstone: your own domain

Companion notebook for the [Capstone](../plan/capstone.md).

**You will:** stub out a domain-specific verifier, wire it into the existing
pipeline, and check the two things that decide whether a domain is viable.

Everything in the reasoning pipeline is domain-agnostic except the verifier. This
notebook is the exercise of proving that to yourself.

## 0 · Setup

Run this cell first. It picks a device and imports everything the notebook needs.

If you get `ModuleNotFoundError: reasoning_from_scratch`, your kernel is not the
course environment — see the [setup page](../start/setup.md).

## 1 · Pick a domain and write the verifier first

Rule: if you cannot write the verifier in an afternoon, pick a different domain.

Fill in `my_grade_answer` below. It must return `True` only when the prediction is
*provably* equivalent to the ground truth.

In [ ]:
def my_grade_answer(pred_text, gt_text):
    """Return True only if pred is provably equivalent to gt.

    Examples of what this might do, depending on domain:
      - text-to-SQL : execute both, compare result sets
      - units       : parse both, compare with a tolerance
      - regex       : run both against a test corpus
      - code        : run the unit tests
    """
    raise NotImplementedError("write your verifier")

## 2 · Test the verifier before touching a model

Twenty cases, including five you expect to fail. If you cannot pass this, no amount
of training will help — you would be training against a broken reward.

In [ ]:
cases = [
    # (prediction, ground_truth, expected_result)
    ("...", "...", True),
    ("...", "...", False),
]

passed = 0
for pred, gt, expect in cases:
    try:
        got = my_grade_answer(pred, gt)
    except NotImplementedError:
        print("verifier not written yet - fill in section 1")
        break
    passed += (got == expect)
    print(f"{'ok ' if got == expect else '!! '}{pred!r:30} vs {gt!r:20} -> {got} (expect {expect})")
print(f"{passed}/{len(cases)}")

## 3 · The 50-item eval set

Your `math500_test.json` equivalent. Same field names, so the existing tooling
works.

In [ ]:
import json
from pathlib import Path

my_eval = [
    # {"problem": "...", "answer": "...", "level": 1, "subject": "my-domain"},
]

Path("my_eval.json").write_text(json.dumps(my_eval, indent=2))
print(f"wrote {len(my_eval)} items")

## 4 · Baseline, before you change anything

Without this number you cannot tell whether training helped. Reuse the chapter 3
evaluation loop with your own prompt template.

In [ ]:
from reasoning_from_scratch.ch03 import load_model_and_tokenizer, evaluate_math500_stream

def my_render_prompt(problem):
    return (
        "You are a precise assistant.\n"
        "Write the final result on a new line as:\n\\boxed{ANSWER}\n\n"
        f"Question:\n{problem}\n\nAnswer:"
    )

model, tokenizer = load_model_and_tokenizer(
    which_model="base", device=device, use_compile=False
)

# TRY: once my_eval has items, adapt evaluate_math500_stream's data argument
# records = evaluate_math500_stream(model, tokenizer, device, my_eval[:50])
print("baseline pending - add items to my_eval first")

## 5 · Choose your lever

| You have | Do | Chapter |
| --- | --- | --- |
| No GPU, teacher API access | Distil: generate traces, verify, train | [8](../chapters/ch08-distillation.md) |
| GPU, cheap verifier | GRPO from the base model | [6](../chapters/ch06-grpo.md) |
| Neither | Inference scaling only — still a valid result | [4](../chapters/ch04-inference-scaling.md) |

## 6 · The write-up

A short report with: baseline, method, result, failure modes, one chart. The
failure modes are the interesting part — anyone can report a number that went up.

```markdown
## Result
Baseline: X% on 50 items.
Method: GRPO, 200 steps, verifier = <yours>.
Result: Y%.

## What the model learned to do differently
<two or three sentences: not "accuracy improved" but what behaviour changed>

## Failure modes hit
- <what broke, and the metric that showed it>
```

## 7 · Sanity checks before you call it done

- [ ] `python run_eval.py` reproduces your headline number from a clean checkout
- [ ] Your verifier has tests, including cases it must reject
- [ ] Your eval set was never in training data
- [ ] Your training log has the chapter 7 metric columns
- [ ] You can state the baseline, not just the result